# ADY201m Assignment - Part 1: SQL queries on CARDIO_TRAIN
Do not create IBM Cloud account, **SQLite** (Python `sqlite3`) is used instead of IBM Db2 + `ibm_db`. SQL statements are the same as in the assignment.

In [1]:
import sqlite3, pandas as pd

df = pd.read_csv("ADY201m-Assignment- Dataset-cardio_train_raw.csv", sep=";")
df.columns = [c.upper() for c in df.columns]
df["AGE"] = (df["AGE"] // 365).astype(int)   # days -> years (as in the PDF)
df.head()

,ID,AGE,GENDER,HEIGHT,WEIGHT,AP_HI,AP_LO,CHOLESTEROL,GLUC,SMOKE,ALCO,ACTIVE,CARDIO
0,0,50,2,168,62.0,110,80,1,1,0,0,1,0
1,1,55,1,156,85.0,140,90,3,1,0,0,1,1
2,2,51,1,165,64.0,130,70,3,1,0,0,0,1
3,3,48,2,169,82.0,150,100,1,1,0,0,1,1
4,4,47,1,156,56.0,100,60,1,1,0,0,0,0


## Load data into the database (equivalent to uploading CSV to Db2)

In [2]:
conn = sqlite3.connect("cardio.db")
df.to_sql("CARDIO_TRAIN", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT * FROM CARDIO_TRAIN LIMIT 5", conn))

   ID  AGE  GENDER  HEIGHT  WEIGHT  AP_HI  AP_LO  CHOLESTEROL  GLUC  SMOKE  \
0   0   50       2     168    62.0    110     80            1     1      0   
1   1   55       1     156    85.0    140     90            3     1      0   
2   2   51       1     165    64.0    130     70            3     1      0   
3   3   48       2     169    82.0    150    100            1     1      0   
4   4   47       1     156    56.0    100     60            1     1      0   

   ALCO  ACTIVE  CARDIO  
0     0       1       0  
1     0       1       1  
2     0       0       1  
3     0       1       1  
4     0       0       0  


## Helper: run a query and count how many rows have CARDIO = 1

In [3]:
def top10(query, col):
    cur = conn.execute(query)
    rows = cur.fetchall()
    count = 0
    for v, cardio in rows:
        if cardio == 1:
            count += 1
        print(f" {col}: {float(v)} CARDIO: {cardio}")
    print(f"There are {count}/10 cases got illness ({count*10}%).")

## Task 1: Number of rows

In [4]:
print(conn.execute("SELECT COUNT(ID) FROM CARDIO_TRAIN").fetchone()[0])

70000


## Task 2: Number of cases with cardiovascular disease

In [5]:
print(conn.execute("SELECT COUNT(CARDIO) FROM CARDIO_TRAIN WHERE CARDIO=1").fetchone()[0])

34979


## Task 3: Women and men

In [6]:
print("GENDER=1:", conn.execute("SELECT COUNT(GENDER) FROM CARDIO_TRAIN WHERE GENDER=1").fetchone()[0])
print("GENDER=2:", conn.execute("SELECT COUNT(GENDER) FROM CARDIO_TRAIN WHERE GENDER=2").fetchone()[0])

GENDER=1: 45530
GENDER=2: 24470


## Task 4: Top 10 highest AP_HI - % ill

In [7]:
top10("SELECT AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC LIMIT 10", "AP_HI")

 AP_HI: 16020.0 CARDIO: 1
 AP_HI: 14020.0 CARDIO: 1
 AP_HI: 14020.0 CARDIO: 1
 AP_HI: 14020.0 CARDIO: 1
 AP_HI: 14020.0 CARDIO: 0
 AP_HI: 13010.0 CARDIO: 1
 AP_HI: 13010.0 CARDIO: 0
 AP_HI: 11500.0 CARDIO: 1
 AP_HI: 11020.0 CARDIO: 1
 AP_HI: 2000.0 CARDIO: 1
There are 8/10 cases got illness (80%).


## Task 5: 10 people with weight above average - % ill

In [8]:
top10("""SELECT WEIGHT, CARDIO FROM CARDIO_TRAIN
WHERE WEIGHT > (SELECT AVG(WEIGHT) FROM CARDIO_TRAIN)
ORDER BY WEIGHT DESC LIMIT 10""", "WEIGHT")

 WEIGHT: 200.0 CARDIO: 0
 WEIGHT: 200.0 CARDIO: 1
 WEIGHT: 183.0 CARDIO: 1
 WEIGHT: 181.0 CARDIO: 1
 WEIGHT: 180.0 CARDIO: 1
 WEIGHT: 180.0 CARDIO: 1
 WEIGHT: 180.0 CARDIO: 1
 WEIGHT: 180.0 CARDIO: 1
 WEIGHT: 178.0 CARDIO: 0
 WEIGHT: 178.0 CARDIO: 1
There are 8/10 cases got illness (80%).


## Task 6: GLUC > 1 (10 rows)

In [9]:
top10("SELECT GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC > 1 LIMIT 10", "GLUC")

 GLUC: 2.0 CARDIO: 0
 GLUC: 3.0 CARDIO: 1
 GLUC: 3.0 CARDIO: 0
 GLUC: 2.0 CARDIO: 1
 GLUC: 3.0 CARDIO: 0
 GLUC: 3.0 CARDIO: 1
 GLUC: 2.0 CARDIO: 0
 GLUC: 2.0 CARDIO: 1
 GLUC: 2.0 CARDIO: 0
 GLUC: 2.0 CARDIO: 1
There are 5/10 cases got illness (50%).


## Task 7: 10 oldest people

In [10]:
top10("SELECT AGE, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC LIMIT 10", "AGE")

 AGE: 64.0 CARDIO: 0
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 0
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 0
 AGE: 64.0 CARDIO: 1
 AGE: 64.0 CARDIO: 1
There are 7/10 cases got illness (70%).


## Task 8: CHOLESTEROL = 3 (10 rows)

In [11]:
top10("SELECT CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL=3 LIMIT 10", "CHOLESTEROL")

 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 0
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
 CHOLESTEROL: 3.0 CARDIO: 1
There are 9/10 cases got illness (90%).


## Task 9: 10 smokers

In [12]:
top10("SELECT SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE=1 LIMIT 10", "SMOKE")

 SMOKE: 1.0 CARDIO: 0
 SMOKE: 1.0 CARDIO: 0
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 0
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 1
 SMOKE: 1.0 CARDIO: 1
There are 7/10 cases got illness (70%).


## Task 10: 10 people who exercise

In [13]:
top10("SELECT ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE=1 LIMIT 10", "ACTIVE")

 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 1
 ACTIVE: 1.0 CARDIO: 1
 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 1
 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 0
 ACTIVE: 1.0 CARDIO: 0
There are 3/10 cases got illness (30%).


## Extra queries from the console part (AP_LO lowest, alcohol)

In [14]:
print(pd.read_sql("SELECT * FROM CARDIO_TRAIN ORDER BY AP_LO ASC LIMIT 10", conn))
print(pd.read_sql("SELECT * FROM CARDIO_TRAIN WHERE ALCO=1 LIMIT 10", conn))

      ID  AGE  GENDER  HEIGHT  WEIGHT  AP_HI  AP_LO  CHOLESTEROL  GLUC  SMOKE  \
0  85816   61       1     167    74.0     15    -70            1     1      0   
1   2845   62       2     167    59.0    906      0            1     1      0   
2  19258   41       2     150    60.0    130      0            1     1      0   
3  23512   56       1     156    50.0    138      0            3     1      0   
4  24837   49       1     164    78.0    138      0            1     1      0   
5  32749   58       2     166    68.0    149      0            1     1      0   
6  34120   44       1     161    92.0    906      0            2     1      0   
7  36325   49       2     176    73.0     12      0            1     1      0   
8  39577   64       2     180    73.0    140      0            1     1      1   
9  45400   44       1     170    64.0    907      0            1     1      0   

   ALCO  ACTIVE  CARDIO  
0     0       1       1  
1     0       1       0  
2     0       0       0  
3   

## Export table to CSV and close connection

In [15]:
out = pd.read_sql("SELECT * FROM CARDIO_TRAIN", conn)
out.to_csv("CARDIO_TRAIN.csv", index=False)
conn.close()
print(out.shape)

(70000, 13)


## (Optional) Original IBM Db2 connection code - fill in your own credentials if you have Db2
```python
import ibm_db
dsn = ("DRIVER={IBM DB2 ODBC DRIVER};DATABASE=bludb;HOSTNAME=<host>;PORT=<port>;"
       "PROTOCOL=TCPIP;UID=<uid>;PWD=<pwd>;SECURITY=SSL;")
conn = ibm_db.connect(dsn, "", "")
res = ibm_db.exec_immediate(conn, "SELECT COUNT(ID) FROM CARDIO_TRAIN")
print(ibm_db.fetch_both(res))
```